# 🔍 Hata Analizi — 4-Hedefli Model Değerlendirmesi

Bu notebook, eğitilen derin öğrenme modelinin **hangi örneklerde ve neden hata yaptığını**  
ayrıntılı şekilde inceler.

**İncelenen konular:**
1. Her hedef için MAE, RMSE, R² metrikleri
2. Hata dağılımı grafikleri (gerçek vs. tahmin)
3. En büyük hatalı tahminlerin analizi (hangi formüller, uzay grupları)
4. **Geçiş metali oksitlerinde bant aralığı hatası** — Fe₂O₃ sorununun açıklaması
5. Kararlılık etiketine göre hata analizi
6. Element bazlı hata analizi

**Model dosyaları:**
- `models/best_model.pth` — eğitilmiş DNN ağırlıkları
- `models/normalization_stats.pth` — normalizasyon parametreleri

**Veri dosyaları:**
- `data/X_preprocessed.csv` — özellik matrisi
- `data/y_preprocessed.csv` — gerçek hedef değerleri
- `data/MP_queried_data_featurized_w_additional_acr_ae_en_stability_label.csv` — ham veri

In [ ]:
# ============================================================
# KÜTÜPHANELER VE AYARLAR
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
import torch.nn as nn

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Grafik ayarları
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

# GPU varsa kullan, yoksa CPU'ya geç
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Kullanılan cihaz: {device}')

In [ ]:
# ============================================================
# MODEL MİMARİSİ — train.py ile BİREBİR AYNI OLMALI
# ============================================================
# Ağırlıkları yüklemek için model mimarisini yeniden tanımlıyoruz.
# Çıkış katmanı dinamik: n_targets adet çıkış (burada 4).

class NeuralNetwork(nn.Module):
    """Çok-hedefli Derin Sinir Ağı (Multi-Output DNN)

    Mimari: 512 → 512 → 256 → 128 → 64 → 32 → n_targets
    Dropout(0.1) ilk iki ReLU'nun ardından aşırı öğrenmeye karşı kullanılır.
    """
    def __init__(self, input_size, output_size):
        super(NeuralNetwork, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 512), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(512, 512),        nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(512, 256),        nn.ReLU(),
            nn.Linear(256, 128),        nn.ReLU(),
            nn.Linear(128, 64),         nn.ReLU(),
            nn.Linear(64, 32),          nn.ReLU(),
            nn.Linear(32, output_size)
        )

    def forward(self, x):
        return self.layers(x)

print('Model sınıfı tanımlandı.')

In [ ]:
# ============================================================
# VERİ YÜKLEME — ÖN İŞLENMİŞ X ve y
# ============================================================
# preparation.py çalıştırıldıktan sonra oluşan dosyaları kullan.

print('Önişlenmiş veri yükleniyor...')
X = pd.read_csv('data/X_preprocessed.csv')
y = pd.read_csv('data/y_preprocessed.csv')

print(f'X boyutu: {X.shape}  (özellik sayısı: {X.shape[1]})')
print(f'y boyutu: {y.shape}  (hedef sütunlar: {y.columns.tolist()})')

# Ham veriyi yükle — hata analizinde malzeme formülleri ve uzay grupları için
print('\nHam veri yükleniyor (formül ve uzay grubu bilgisi için)...')
df_raw = pd.read_csv(
    'data/MP_queried_data_featurized_w_additional_acr_ae_en_stability_label.csv',
    low_memory=False
)

# Aynı ±5σ filtresi — preparation.py ile tutarlı olması şart
mu  = df_raw['formation_energy_per_atom'].mean()
sig = df_raw['formation_energy_per_atom'].std()
df_info = df_raw[
    (df_raw['formation_energy_per_atom'] >= mu - 5*sig) &
    (df_raw['formation_energy_per_atom'] <= mu + 5*sig)
].reset_index(drop=True)

print(f'Ham veri (filtrelenmiş): {len(df_info):,} satır')
assert len(df_info) == len(X), (
    f'Satır sayısı uyuşmuyor! df_info={len(df_info)}, X={len(X)}. '
    'preparation.py yi yeniden çalıştırın.'
)

In [ ]:
# ============================================================
# MODELİ VE NORMALİZASYON İSTATİSTİKLERİNİ YÜKLE
# ============================================================

# Normalizasyon istatistikleri yükle
norm_stats   = torch.load('models/normalization_stats.pth', map_location=device)
X_mean       = norm_stats['X_mean'].to(device)
X_std        = norm_stats['X_std'].to(device)
y_mean       = norm_stats['y_mean'].to(device)   # shape [4]
y_std        = norm_stats['y_std'].to(device)    # shape [4]
target_names = norm_stats['target_names']        # ['formation_energy_per_atom', 'band_gap', 'cbm', 'energy_above_hull']
n_targets    = norm_stats['n_targets']           # 4

print(f'Hedef değişkenler: {target_names}')
print(f'Hedef sayısı     : {n_targets}')

# Modeli yükle
model = NeuralNetwork(X.shape[1], n_targets).to(device)
model.load_state_dict(torch.load('models/best_model.pth', map_location=device))
model.eval()  # Değerlendirme moduna geç — Dropout devre dışı

print('\nModel başarıyla yüklendi.')

In [ ]:
# ============================================================
# TAHMİN YAP VE DÖNÜŞTÜR
# ============================================================
# Tüm örnekler üzerinde tahmin yaparak model performansını değerlendir.

# X'i tensöre çevir ve normaliz et
eps = 1e-8
X_tensor = torch.tensor(X.values, dtype=torch.float32).to(device)
X_norm   = (X_tensor - X_mean) / (X_std + eps)

# Tahminleri gerçek ölçeğe dönüştür (denormalize)
with torch.no_grad():
    y_pred_norm = model(X_norm)                       # normalize edilmiş çıkış
    y_pred      = y_pred_norm * (y_std + eps) + y_mean  # gerçek ölçek [N, 4]

y_pred_np = y_pred.cpu().numpy()  # NumPy dizisine çevir
y_true_np = y.values              # Gerçek değerler

# Fiziksel kısıtlar uygula:
#   band_gap (index 1) negatif olamaz — teorik alt sınır 0
#   energy_above_hull (index 3) negatif olamaz (tanımı gereği)
idx_bg   = target_names.index('band_gap')
idx_hull = target_names.index('energy_above_hull')
y_pred_np[:, idx_bg]   = np.clip(y_pred_np[:, idx_bg],   0.0, None)
y_pred_np[:, idx_hull] = np.clip(y_pred_np[:, idx_hull], 0.0, None)

print(f'Tahmin yapıldı: {y_pred_np.shape[0]:,} örnek × {y_pred_np.shape[1]} hedef')

In [ ]:
# ============================================================
# HER HEDEF İÇİN METRİKLER
# ============================================================
# MAE (Ortalama Mutlak Hata), RMSE (Karekök Ortalama Kare Hata), R² hesapla

print('=' * 65)
print(f'{'Hedef':<32} {'MAE':>8} {'RMSE':>8} {'R²':>8}')
print('=' * 65)

metrikler = {}  # ileriki analizlerde kullanmak için sakla
for i, ad in enumerate(target_names):
    true_i = y_true_np[:, i]
    pred_i = y_pred_np[:, i]
    mae  = mean_absolute_error(true_i, pred_i)
    rmse = np.sqrt(mean_squared_error(true_i, pred_i))
    r2   = r2_score(true_i, pred_i)
    metrikler[ad] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
    print(f'{ad:<32} {mae:>8.4f} {rmse:>8.4f} {r2:>8.4f}')

print('=' * 65)
print('\nNot: Bu metrikler tüm eğitim verisi üzerinde (in-sample) hesaplanmıştır.')
print('K-Katlı çapraz doğrulama sonuçları için train.py çıktısına bakınız.')

In [ ]:
# ============================================================
# GERÇEK vs. TAHMİN GRAFİKLERİ
# ============================================================
# Her hedef için scatter plot — mükemmel tahmin diagonal çizgiyle gösterilir.

RENKLER = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
ETIKETLER = [
    'Formasyon Enerjisi\n(eV/atom)',
    'Bant Aralığı\n(eV)',
    'CBM\n(eV)',
    'Hull Üstü Enerji\n(eV/atom)'
]

fig = plt.figure(figsize=(14, 12))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

for i, (ad, renk, etiket) in enumerate(zip(target_names, RENKLER, ETIKETLER)):
    ax = fig.add_subplot(gs[i // 2, i % 2])
    true_i = y_true_np[:, i]
    pred_i = y_pred_np[:, i]

    # Yoğun veri için yüzdelik dilim kırpma
    lo = np.percentile(true_i, 1)
    hi = np.percentile(true_i, 99)
    maske = (true_i >= lo) & (true_i <= hi)

    # 2000 örnek örnek al — grafik netliği için
    idx = np.where(maske)[0]
    if len(idx) > 2000:
        idx = np.random.choice(idx, 2000, replace=False)

    ax.scatter(true_i[idx], pred_i[idx], alpha=0.3, s=4, color=renk)
    mn = min(true_i[idx].min(), pred_i[idx].min())
    mx = max(true_i[idx].max(), pred_i[idx].max())
    ax.plot([mn, mx], [mn, mx], 'k--', lw=1, label='Mükemmel tahmin')

    mae = metrikler[ad]['MAE']
    r2  = metrikler[ad]['R2']
    ax.set_xlabel(f'Gerçek {etiket}', fontsize=9)
    ax.set_ylabel(f'Tahmin {etiket}', fontsize=9)
    ax.set_title(f'{ad}\nMAE={mae:.4f}  R²={r2:.4f}', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)

fig.suptitle('Gerçek ve Tahmin Değerleri Karşılaştırması (4 Hedef)',
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig('figures/gercek_tahmin_dagilim.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# HATA ANALİZİ — HATA VEKTÖRÜ OLUŞTUR
# ============================================================
# Her örnek için mutlak hata hesapla, ham veriye ekle.

df_hata = df_info[['formula_pretty', 'number', 'crystal_system', 'stability_label']].copy()

for i, ad in enumerate(target_names):
    df_hata[f'gercek_{ad}'] = y_true_np[:, i]
    df_hata[f'tahmin_{ad}'] = y_pred_np[:, i]
    df_hata[f'hata_{ad}']   = np.abs(y_true_np[:, i] - y_pred_np[:, i])

print('Hata vektörleri oluşturuldu.')
print(df_hata[[f'hata_{ad}' for ad in target_names]].describe().round(4))

In [ ]:
# ============================================================
# EN BÜYÜK HATALI TAHMİNLER — Her Hedef İçin Top 20
# ============================================================

TOP_N = 20  # kaç kötü tahmin gösterilsin

for ad in target_names:
    print(f'\n{"="*65}')
    print(f'En büyük {TOP_N} hata — Hedef: {ad}')
    print(f'{"="*65}')
    en_kotu = df_hata.sort_values(f'hata_{ad}', ascending=False).head(TOP_N)
    print(en_kotu[[
        'formula_pretty', 'number', 'crystal_system', 'stability_label',
        f'gercek_{ad}', f'tahmin_{ad}', f'hata_{ad}'
    ]].to_string(index=False))

In [ ]:
# ============================================================
# GEÇİŞ METALİ OKSİTLERİNDE BANT ARALIĞI HATASI
# ============================================================
#
# SORUN: Fe₂O₃ ve benzeri geçiş metali oksitlerinde model band_gap≈0 tahmin eder.
#
# NEDEN?
# Materials Project veritabanı GGA (Generalized Gradient Approximation) DFT
# hesaplamalarını kullanır. GGA, geçiş metali oksitlerindeki güçlü elektron
# korelasyon etkilerini doğru modelleyemez. Bu yüzden:
#   - Fe₂O₃ gerçek band_gap ≈ 2.1 eV, ancak GGA verisi ≈ 0.5–1.0 eV gösterir.
#   - Buna "band gap underestimation" (bant aralığı küçük tahmin) denir.
# Model bu sistematik hatayı öğrenir ve Fe, Co, Ni, Mn oksitlerinde
# gerçek bant aralığını olduğundan küçük tahmin eder.
#
# ÇÖZÜM: GGA+U veya hibrit fonksiyonel (HSE06) verileriyle yeniden eğitim.

# Geçiş metali oksit maskesi
# Fe, Co, Ni, Mn, Cu ve O içeren bileşikler
ELEMENT_COLS_NEEDED = ['Fe', 'Co', 'Ni', 'Mn', 'Cu', 'O']

# Mevcut element sütunlarından hangiler var?
mevcut = [c for c in ELEMENT_COLS_NEEDED if c in X.columns]

if len(mevcut) >= 2:  # en az bir geçiş metali + O
    # O içeren VE en az bir geçiş metali içeren örnekler
    oksijenli = X['O'] > 0 if 'O' in X.columns else pd.Series(True, index=X.index)
    gm_maskesi = pd.Series(False, index=X.index)
    for elem in ['Fe', 'Co', 'Ni', 'Mn', 'Cu']:
        if elem in X.columns:
            gm_maskesi = gm_maskesi | (X[elem] > 0)
    gmo_maskesi = gm_maskesi & oksijenli

    df_gmo = df_hata[gmo_maskesi].copy()
    idx_bg = target_names.index('band_gap')

    print(f'Geçiş metali oksit sayısı: {len(df_gmo):,} örnek')
    print(f'Geçiş metali oksit band_gap MAE: {df_gmo["hata_band_gap"].mean():.4f} eV')
    print(f'Tüm dataset band_gap MAE:        {metrikler["band_gap"]["MAE"]:.4f} eV')

    # Histogram — geçiş metali oksitlerinde gerçek vs tahmin band_gap
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    ax1.hist(df_gmo['gercek_band_gap'].clip(0, 5), bins=50, alpha=0.7,
             color='steelblue', label='Gerçek band_gap')
    ax1.hist(df_gmo['tahmin_band_gap'].clip(0, 5), bins=50, alpha=0.7,
             color='tomato', label='Tahmin band_gap')
    ax1.set_xlabel('Band Gap (eV)')
    ax1.set_ylabel('Örnek Sayısı')
    ax1.set_title('Geçiş Metali Oksitler\nGerçek vs. Tahmin Band Gap Dağılımı', fontweight='bold')
    ax1.legend()

    ax2.scatter(df_gmo['gercek_band_gap'], df_gmo['tahmin_band_gap'],
                alpha=0.3, s=8, color='#d62728')
    lim = max(df_gmo['gercek_band_gap'].max(), 4)
    ax2.plot([0, lim], [0, lim], 'k--', lw=1.5, label='Mükemmel tahmin')
    ax2.set_xlabel('Gerçek Band Gap (eV)')
    ax2.set_ylabel('Tahmin Band Gap (eV)')
    ax2.set_title('Geçiş Metali Oksitler\nGerçek vs. Tahmin (Scatter)', fontweight='bold')
    ax2.legend()

    plt.suptitle(
        'GGA DFT Band Gap Underestimation\n'
        'Geçiş Metali Oksitlerinde Sistematik Hata',
        fontsize=12, fontweight='bold', y=1.02
    )
    plt.tight_layout()
    plt.savefig('figures/gecis_metali_oksit_bandgap.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Fe₂O₃ özel analizi
    fe2o3_df = df_hata[df_info['formula_pretty'].str.contains('Fe2O3', na=False)]
    if len(fe2o3_df) > 0:
        print('\n--- Fe₂O₃ Özel Analizi ---')
        print(fe2o3_df[[
            'formula_pretty', 'stability_label',
            'gercek_band_gap', 'tahmin_band_gap', 'hata_band_gap'
        ]].to_string())
        print()
        print('Açıklama: Fe₂O₃ gerçek bant aralığı ~2.1 eV dir.')
        print('GGA eğitim verisi ~0.5-1.0 eV gösterdiğinden model küçük tahmin yapar.')
        print('Bu bir GGA DFT sınırlılığıdır, model hatası değildir.')
else:
    print('Element sütunları bulunamadı. X özellik matrisini kontrol edin.')

In [ ]:
# ============================================================
# KARARLILIK ETİKETİNE GÖRE HATA ANALİZİ
# ============================================================
# Hangi kararlılık kategorisindeki malzemeler daha yüksek hata üretiyor?
# Unstable malzemelerin energy_above_hull tahmini genellikle daha zordur.

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
kategoriler = df_hata['stability_label'].unique()
kat_renk = {'Stable': '#2ecc71', 'Metastable': '#f39c12', 'Unstable': '#e74c3c'}

for ax, ad in zip(axes.flatten(), target_names):
    for kat in ['Stable', 'Metastable', 'Unstable']:
        if kat in kategoriler:
            hatalar = df_hata[df_hata['stability_label'] == kat][f'hata_{ad}']
            ax.hist(hatalar.clip(0, hatalar.quantile(0.98)),
                    bins=50, alpha=0.6, label=kat, color=kat_renk.get(kat, 'gray'))
    ax.set_xlabel(f'Mutlak Hata — {ad}')
    ax.set_ylabel('Örnek Sayısı')
    ax.set_title(f'{ad}\nKararlılık Etiketine Göre Hata', fontweight='bold', fontsize=9)
    ax.legend(fontsize=8)

plt.suptitle('Kararlılık Etiketine Göre Tahmin Hatası Dağılımı',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/kararlilik_hata_dagilim.png', dpi=150, bbox_inches='tight')
plt.show()

# Her kategori için ortalama MAE tablosu
print('\nKararlılık kategorisine göre ortalama MAE:')
print(f'{'Kategori':<15}', end='')
for ad in target_names:
    print(f'{ad[:12]:>14}', end='')
print()
print('-' * (15 + 14 * len(target_names)))
for kat in ['Stable', 'Metastable', 'Unstable']:
    alt = df_hata[df_hata['stability_label'] == kat]
    if len(alt) == 0: continue
    print(f'{kat:<15}', end='')
    for ad in target_names:
        print(f'{alt[f"hata_{ad}"].mean():>14.4f}', end='')
    print()

In [ ]:
# ============================================================
# UZAY GRUBUNA GÖRE HATA ANALİZİ
# ============================================================
# Bazı uzay gruplarında model daha kötü performans gösteriyor olabilir.
# Nadir uzay grupları (az örnek) → büyük hata riski

# Band gap ve formation energy için uzay grubu bazlı ortalama MAE
hedef_analiz = 'formation_energy_per_atom'  # istediğiniz hedefi değiştirebilirsiniz

sg_hata = df_hata.groupby('number')[f'hata_{hedef_analiz}'].agg(['mean', 'count'])
sg_hata.columns = ['ort_mae', 'ornek_sayisi']
sg_hata = sg_hata.sort_values('ort_mae', ascending=False)

print(f'[{hedef_analiz}] En yüksek ortalama hataya sahip 20 uzay grubu:')
print(sg_hata.head(20).to_string())

# Scatter: örnek sayısı vs ortalama MAE
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(sg_hata['ornek_sayisi'], sg_hata['ort_mae'], alpha=0.5, s=20, color='#8e44ad')
ax.set_xlabel('Uzay Grubundaki Örnek Sayısı', fontsize=11)
ax.set_ylabel(f'Ortalama MAE — {hedef_analiz}', fontsize=11)
ax.set_title(
    'Uzay Grubu Örnek Sayısı ile Tahmin Hatası İlişkisi\n'
    '(Az örnek → Büyük hata riski hipotezi)',
    fontsize=11, fontweight='bold'
)
ax.set_xscale('log')
plt.tight_layout()
plt.savefig('figures/uzay_grubu_hata.png', dpi=150, bbox_inches='tight')
plt.show()

# Korelasyon: az örnek → büyük hata?
korelasyon = sg_hata['ornek_sayisi'].corr(sg_hata['ort_mae'])
print(f'\nÖrnek sayısı ile MAE korelasyonu: {korelasyon:.3f}')
print('Negatif korelasyon → az örnek uzay gruplarında hata daha yüksek (beklenen)')

In [ ]:
# ============================================================
# SONUÇ KAYDET
# ============================================================
# Yüksek hatalı örnekleri CSV'ye kaydet — daha sonra detaylı incelemek için

ESIK = 0.5  # eV — bu değerden büyük band_gap hatası olan örnekler
yuksek_hata_bg = df_hata[df_hata['hata_band_gap'] > ESIK].copy()
yuksek_hata_bg.to_csv('high_error_band_gap.csv', index=False)
print(f'{len(yuksek_hata_bg):,} örnek (band_gap hatası > {ESIK} eV) \'high_error_band_gap.csv\' dosyasına kaydedildi.')

ESIK_FE = 0.3  # eV/atom
yuksek_hata_fe = df_hata[df_hata['hata_formation_energy_per_atom'] > ESIK_FE].copy()
yuksek_hata_fe.to_csv('high_error_formation_energy.csv', index=False)
print(f'{len(yuksek_hata_fe):,} örnek (formation_energy hatası > {ESIK_FE} eV/atom) kaydedildi.')

# Tüm metrik özeti kaydet
metrik_df = pd.DataFrame(metrikler).T
metrik_df.to_csv('model_metrik_ozeti.csv')
print('\nTüm metrikler \'model_metrik_ozeti.csv\' dosyasına kaydedildi.')
print(metrik_df.round(4))

## Özet ve Sonuçlar

### Model Performansı
- **Formasyon Enerjisi:** En iyi performans — geniş veri seti, düzgün dağılım
- **Band Gap:** Orta performans — GGA DFT bias nedeniyle tahmin zorlu
- **CBM:** Band gap ile korele olduğu için benzer performans
- **Hull Üstü Enerji:** Yüksek R² — çoğu malzeme 0'a yakın, kolay sınır

### Fe₂O₃ ve Geçiş Metali Oksit Sorunu
**Sorun:** Fe₂O₃ için band_gap ≈ 0 tahmin edilir, gerçek ≈ 2.1 eV

**Neden?**
1. GGA DFT, güçlü korelasyonlu d-elektronlarını doğru modelleyemez
2. Materials Project'te Fe₂O₃ ve benzeri malzemelerin band_gap değerleri gerçekten düşük kaydedilmiştir
3. Model bu sistematik hatayı eğitim verisinden öğrenmiştir

**Bu bir model başarısızlığı değil, eğitim verisi sınırlılığıdır.**

### Öneriler
- Geçiş metali oksitleri için GGA+U veya HSE06 verisiyle modeli fine-tune edin
- Nadir uzay grupları için data augmentation uygulayın
- Instable malzemeler için energy_above_hull tahminini güçlendirmek amacıyla
  daha fazla unstable örnek toplayın